# 🦜 LangChain, LangGraph & Agentic AI — Complete Worksheet

**Topics Covered:**
- Part 1: LangChain Fundamentals (LLMs, Chains, Prompts, Memory, Tools)
- Part 2: LangGraph Fundamentals (Nodes, Edges, State, Conditional Routing)
- Part 3: Agentic AI (ReAct Agents, Multi-Agent Systems, Tool-Using Agents)

> **Setup Note:** This notebook uses `langchain`, `langgraph`, and `ollama`. Run the install cell below. Make sure Ollama is installed and running locally (`ollama serve`).


---
## ⚙️ Installation & Setup

pip install jupyter ipykernel


python -m ipykernel install --user --name=venv --display-name="Python (venv)"

In [ ]:
# Install all required libraries
pip install -q langchain langchain-ollama langchain-community langgraph
print("✅ All packages installed!")

In [ ]:
# Using Ollama with llama3.1:8b — make sure Ollama is running: `ollama serve`
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1:8b", temperature=0)
response = llm.invoke("Say hello in one sentence.")
print(response.content)

---
# 📘 PART 1 — LangChain Fundamentals

LangChain is a framework for building LLM-powered applications. The core building blocks are:

| Component | What it does |
|-----------|-------------|
| **LLM / ChatModel** | The language model itself |
| **PromptTemplate** | Reusable, parameterized prompts |
| **Chain** | A sequence of components connected together |
| **Memory** | Stores conversation history |
| **Tool** | A function the LLM can call |
| **Retriever** | Fetches relevant documents for RAG |

LangChain's modern API uses **LCEL (LangChain Expression Language)** — connecting components with the `|` pipe operator.

## 1.1 — PromptTemplates

A `PromptTemplate` lets you define a reusable prompt with **placeholders** `{like_this}` that get filled in at runtime.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# Define a prompt with a {topic} placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert teacher. Explain concepts simply."),
    ("human", "Explain {topic} in 3 bullet points for a beginner.")
])

# Format the prompt — this creates the actual message list
formatted = prompt.format_messages(topic="neural networks")
for msg in formatted:
    print(f"[{msg.type.upper()}]: {msg.content}")

In [ ]:
# Now pipe prompt -> LLM and invoke
chain = prompt | llm
response = chain.invoke({"topic": "neural networks"})
print(response.content)

## 1.2 — Output Parsers

By default, LLMs return a message object. **Output parsers** transform the output into whatever format you need: plain text, JSON, lists, Pydantic models, etc.

In [ ]:
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import PromptTemplate

# --- StrOutputParser: extracts just the text string ---
str_chain = prompt | llm | StrOutputParser()
text = str_chain.invoke({"topic": "transformers"})
print(type(text))   # str, not AIMessage
print(text[:200])

In [ ]:
# --- JsonOutputParser: forces the LLM to return valid JSON ---
json_prompt = PromptTemplate.from_template(
    """Return a JSON object with keys 'term', 'definition', 'example' for: {concept}
    Respond ONLY with valid JSON, no extra text."""
)

json_chain = json_prompt | llm | JsonOutputParser()
result = json_chain.invoke({"concept": "gradient descent"})
print(type(result))  # dict!
print(result)

## 1.3 — Chains (LCEL Pipelines)

Chains connect multiple components. LCEL uses the `|` operator — just like Unix pipes.

```
input → prompt → llm → output_parser → result
```

You can also use `RunnablePassthrough` and `RunnableLambda` to add custom steps.

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# --- Step 1: Summarize a concept ---
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize this concept in one sentence: {concept}"
)

# --- Step 2: Then generate a quiz question from the summary ---
quiz_prompt = ChatPromptTemplate.from_template(
    "Write one multiple-choice quiz question about: {summary}"
)

# Chain: concept -> summarize -> quiz question
summarize_chain = summarize_prompt | llm | StrOutputParser()

full_chain = (
    {"concept": RunnablePassthrough()}     # pass input through
    | summarize_prompt
    | llm
    | StrOutputParser()
    | (lambda summary: {"summary": summary})  # reshape for next prompt
    | quiz_prompt
    | llm
    | StrOutputParser()
)

quiz_q = full_chain.invoke("attention mechanism in transformers")
print(quiz_q)

## 1.4 — Conversation Memory

LLMs are **stateless** — they don't remember previous turns. LangChain's memory components fix this by storing and injecting conversation history.

In [ ]:
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

# A simple chatbot with memory
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor."),
    ("placeholder", "{chat_history}"),  # <- history gets injected here
    ("human", "{question}")
])

base_chain = chat_prompt | llm | StrOutputParser()

# In-memory store: session_id -> history
store = {}

def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chatbot = RunnableWithMessageHistory(
    base_chain,
    get_session_history,
    input_messages_key="question",
    history_messages_key="chat_history"
)

config = {"configurable": {"session_id": "student_1"}}

# Turn 1
r1 = chatbot.invoke({"question": "What is backpropagation?"}, config=config)
print("Turn 1:", r1[:200])

# Turn 2 — refers to previous answer
r2 = chatbot.invoke({"question": "Can you give a simple analogy for what you just explained?"}, config=config)
print("\nTurn 2:", r2[:300])

## 1.5 — Tools & Tool Calling

Tools let LLMs call **external functions** — APIs, databases, calculators, search engines, etc. The LLM decides WHEN to call a tool and with WHAT arguments.

In [ ]:
from langchain_core.tools import tool

# Define tools using the @tool decorator
@tool
def get_word_count(text: str) -> int:
    """Count the number of words in a text string."""
    return len(text.split())

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city. Returns a weather description."""
    # Mock implementation — in real life, call a weather API
    mock_data = {
        "lahore": "Hot and sunny, 38°C",
        "london": "Cloudy with light rain, 14°C",
        "new york": "Clear skies, 22°C"
    }
    return mock_data.get(city.lower(), f"Weather data not available for {city}")

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. E.g., '2 + 3 * 4'"""
    try:
        result = eval(expression)  # Safe for demo; use a proper parser in production
        return str(result)
    except Exception as e:
        return f"Error: {e}"

tools = [get_word_count, get_weather, calculate]

# Bind tools to the LLM
llm_with_tools = llm.bind_tools(tools)

# Ask something that requires a tool
response = llm_with_tools.invoke("What is 347 * 18?")
print("Tool calls requested:", response.tool_calls)

In [ ]:
from langchain_core.messages import ToolMessage

# Execute the tool calls and send results back
tool_map = {t.name: t for t in tools}

messages = [response]
for tc in response.tool_calls:
    result = tool_map[tc["name"]].invoke(tc["args"])
    messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

# Final LLM call with tool result
final = llm_with_tools.invoke(messages)
print("Final answer:", final.content)

---
## 🧠 Part 1 — Exercises

**Exercise 1.1:** Create a `ChatPromptTemplate` that takes a `language` and `concept` as input and asks the LLM to explain the concept in that language. Test it with `language="Urdu"` and `concept="machine learning"`.

**Exercise 1.2:** Build an LCEL chain that: (1) Takes a student's essay, (2) Summarizes it in one paragraph, (3) Gives 2 improvement suggestions.

**Exercise 1.3:** Define a custom `@tool` called `celsius_to_fahrenheit(temp: float)` that converts temperature. Bind it to the LLM and ask: *"What is 100°C in Fahrenheit?"*

**Exercise 1.4 (Challenge):** Build a multi-turn chatbot that remembers the student's name and favorite subject from the first message and refers to them in later turns.

In [ ]:
# Your Exercise 1.1 code here


In [ ]:
# Your Exercise 1.2 code here


In [ ]:
# Your Exercise 1.3 code here


In [ ]:
# Your Exercise 1.4 (Challenge) code here


---
# 📗 PART 2 — LangGraph Fundamentals

## What is LangGraph?

LangChain **chains** are linear pipelines: `A → B → C`. But real-world AI systems need **loops, branches, and state** — think of a research agent that decides whether to search again or stop.

**LangGraph** models AI workflows as a **graph** of nodes and edges:

```
      ┌─────────────┐
      │   Node A    │
      └──────┬──────┘
             │
    ┌────────▼────────┐
    │  Conditional    │  <-- routing logic
    │     Edge        │
    └────┬───────┬────┘
         │       │
       Yes       No
         │       │
      Node B   Node C
```

| Concept | Meaning |
|---------|--------|
| **State** | A shared dict that flows through all nodes |
| **Node** | A Python function that reads/writes state |
| **Edge** | A connection between nodes (fixed or conditional) |
| **START / END** | Special nodes marking entry and exit points |
| **StateGraph** | The graph object you compile and run |

## 2.1 — Hello LangGraph: A Simple 2-Node Graph

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# --- Step 1: Define the State ---
# State is a TypedDict — think of it as the shared memory of the graph
class SimpleState(TypedDict):
    input_text: str
    word_count: int
    summary: str

# --- Step 2: Define Nodes ---
# Each node is a function that takes state and returns a partial update

def count_words(state: SimpleState) -> dict:
    """Node 1: Count words in the input text."""
    count = len(state["input_text"].split())
    print(f"  [count_words] Counted {count} words")
    return {"word_count": count}

def summarize_text(state: SimpleState) -> dict:
    """Node 2: Summarize the input text using LLM."""
    response = llm.invoke(f"Summarize in one sentence: {state['input_text']}")
    print(f"  [summarize_text] Summary generated")
    return {"summary": response.content}

# --- Step 3: Build the Graph ---
builder = StateGraph(SimpleState)

# Add nodes
builder.add_node("count_words", count_words)
builder.add_node("summarize_text", summarize_text)

# Add edges: START -> count_words -> summarize_text -> END
builder.add_edge(START, "count_words")
builder.add_edge("count_words", "summarize_text")
builder.add_edge("summarize_text", END)

# --- Step 4: Compile & Run ---
graph = builder.compile()

initial_state = {
    "input_text": "Artificial intelligence is transforming every industry. Machine learning models can now recognize images, translate languages, and generate text better than ever before.",
    "word_count": 0,
    "summary": ""
}

result = graph.invoke(initial_state)
print("\n✅ Final State:")
print(f"  Word count: {result['word_count']}")
print(f"  Summary: {result['summary']}")

## 2.2 — Conditional Edges (Branching)

Conditional edges let the graph take different paths based on the current state. This is what makes LangGraph powerful — it can make **decisions**.

In [ ]:
from typing import Literal

# A graph that classifies text and routes to different handlers
class ClassifyState(TypedDict):
    text: str
    category: str   # "question", "complaint", or "compliment"
    response: str

# Node: classify the input
def classify_input(state: ClassifyState) -> dict:
    prompt = f"""Classify this customer message into exactly one category:
    - question
    - complaint  
    - compliment
    
    Message: {state['text']}
    Respond with ONLY the single word category."""
    
    category = llm.invoke(prompt).content.strip().lower()
    print(f"  [classify] Category: {category}")
    return {"category": category}

# Three different handler nodes
def handle_question(state: ClassifyState) -> dict:
    resp = llm.invoke(f"Answer this question helpfully: {state['text']}")
    return {"response": f"[ANSWER] {resp.content}"}

def handle_complaint(state: ClassifyState) -> dict:
    resp = llm.invoke(f"Respond empathetically to this complaint: {state['text']}")
    return {"response": f"[APOLOGY] {resp.content}"}

def handle_compliment(state: ClassifyState) -> dict:
    resp = llm.invoke(f"Thank the customer for: {state['text']}")
    return {"response": f"[THANKS] {resp.content}"}

# Routing function — this decides which branch to take
def route_by_category(state: ClassifyState) -> Literal["handle_question", "handle_complaint", "handle_compliment"]:
    category = state["category"]
    if "question" in category:
        return "handle_question"
    elif "complaint" in category:
        return "handle_complaint"
    else:
        return "handle_compliment"

# Build graph
builder = StateGraph(ClassifyState)
builder.add_node("classify_input", classify_input)
builder.add_node("handle_question", handle_question)
builder.add_node("handle_complaint", handle_complaint)
builder.add_node("handle_compliment", handle_compliment)

builder.add_edge(START, "classify_input")

# Conditional edge: from classify_input, use route_by_category to decide
builder.add_conditional_edges(
    "classify_input",
    route_by_category,
    {  # mapping: return value -> node name
        "handle_question": "handle_question",
        "handle_complaint": "handle_complaint",
        "handle_compliment": "handle_compliment"
    }
)
builder.add_edge("handle_question", END)
builder.add_edge("handle_complaint", END)
builder.add_edge("handle_compliment", END)

graph = builder.compile()

# Test with different inputs
for msg in [
    "How do I reset my password?",
    "This product is broken and I want a refund!",
    "Amazing service, I love your team!"
]:
    print(f"\n📨 Input: {msg}")
    result = graph.invoke({"text": msg, "category": "", "response": ""})
    print(f"   {result['response'][:150]}")

## 2.3 — Loops in Graphs (Iterative Refinement)

LangGraph supports **cycles** — a node can loop back to an earlier node. This enables iterative workflows like "keep trying until good enough."

In [ ]:
from typing import Annotated
import operator

# A graph that generates and improves a piece of writing
class WritingState(TypedDict):
    topic: str
    draft: str
    feedback: str
    iteration: int
    max_iterations: int
    final: bool

def generate_draft(state: WritingState) -> dict:
    """Generate or improve the draft."""
    if state["iteration"] == 0:
        prompt = f"Write a short 3-sentence paragraph about: {state['topic']}"
    else:
        prompt = f"""Improve this paragraph based on feedback:
        
        Original: {state['draft']}
        Feedback: {state['feedback']}
        
        Write an improved version."""
    
    draft = llm.invoke(prompt).content
    print(f"  [generate] Iteration {state['iteration'] + 1} draft written")
    return {"draft": draft, "iteration": state["iteration"] + 1}

def evaluate_draft(state: WritingState) -> dict:
    """Evaluate quality and decide if done."""
    prompt = f"""Rate this paragraph quality from 1-10 and give one specific improvement tip.
    Paragraph: {state['draft']}
    Format: SCORE: X\nTIP: <one sentence tip>"""
    
    feedback_text = llm.invoke(prompt).content
    print(f"  [evaluate] Feedback: {feedback_text[:100]}")
    
    # Extract score
    try:
        score = int(feedback_text.split("SCORE:")[1].split("\n")[0].strip())
    except:
        score = 5
    
    # Stop if score >= 8 or max iterations reached
    is_final = score >= 8 or state["iteration"] >= state["max_iterations"]
    return {"feedback": feedback_text, "final": is_final}

def should_continue(state: WritingState) -> Literal["generate_draft", END]:
    """Route: loop back to improve, or exit."""
    if state["final"]:
        return END
    return "generate_draft"

# Build graph
builder = StateGraph(WritingState)
builder.add_node("generate_draft", generate_draft)
builder.add_node("evaluate_draft", evaluate_draft)

builder.add_edge(START, "generate_draft")
builder.add_edge("generate_draft", "evaluate_draft")
builder.add_conditional_edges("evaluate_draft", should_continue)  # <-- loop or exit

graph = builder.compile()

result = graph.invoke({
    "topic": "the importance of sleep for learning",
    "draft": "",
    "feedback": "",
    "iteration": 0,
    "max_iterations": 3,
    "final": False
})

print(f"\n✅ Final draft after {result['iteration']} iteration(s):")
print(result["draft"])

## 2.4 — Built-in Memory with Checkpointers

LangGraph has built-in **checkpointing** — it can save and restore state across invocations, enabling persistent conversations.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from typing import Sequence

class ChatState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

def chatbot_node(state: ChatState) -> dict:
    """Simple chatbot node."""
    system_prompt = "You are a helpful AI tutor. Remember context from earlier in the conversation."
    
    response = llm.invoke(
        [{"role": "system", "content": system_prompt}]
        + list(state["messages"])
    )
    return {"messages": [response]}

# MemorySaver persists state between calls
checkpointer = MemorySaver()

builder = StateGraph(ChatState)
builder.add_node("chatbot", chatbot_node)
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

graph = builder.compile(checkpointer=checkpointer)

# Thread ID identifies this conversation
config = {"configurable": {"thread_id": "session_42"}}

def chat(user_message: str):
    result = graph.invoke(
        {"messages": [HumanMessage(content=user_message)]},
        config=config
    )
    return result["messages"][-1].content

print("Turn 1:", chat("Hi! My name is Ahmed and I'm learning about LLMs."))
print("\nTurn 2:", chat("What's a good analogy for attention mechanisms?"))
print("\nTurn 3:", chat("Can you remind me what I said my name was?"))  # Tests memory!

---
## 🧠 Part 2 — Exercises

**Exercise 2.1:** Build a LangGraph with two nodes: `translate_to_urdu` and `count_characters`. The graph should take an English sentence, translate it, then count how many characters the Urdu translation has.

**Exercise 2.2:** Extend the classify graph from 2.2 to handle a 4th category: `"spam"`. Add a `handle_spam` node that returns a standard rejection message.

**Exercise 2.3:** Build a loop graph that generates a quiz question, then checks if it's clear. If the LLM rates clarity < 7/10, regenerate. Stop at 4 iterations max.

**Exercise 2.4 (Challenge):** Build a fact-checking graph: Node 1 extracts a claim from a sentence. Node 2 asks the LLM if the claim is True/False/Uncertain. Node 3 provides evidence. Use conditional edges based on the verdict.

In [ ]:
# Your Exercise 2.1 code here


In [ ]:
# Your Exercise 2.2 code here


In [ ]:
# Your Exercise 2.3 code here


In [ ]:
# Your Exercise 2.4 (Challenge) code here


---
# 📕 PART 3 — Agentic AI

## What is an Agent?

An **agent** is an AI system that:
1. **Perceives** its environment (inputs, tool results)
2. **Reasons** about what to do next
3. **Acts** using tools or sub-agents
4. **Loops** until the task is complete

### ReAct Pattern
The most common agent architecture is **ReAct** (Reason + Act):
```
Thought: I need to find X
Action: search("X")
Observation: [result]
Thought: Now I need Y
Action: calculate(Y)
Observation: [result]
Thought: I have enough to answer
Final Answer: ...
```

### Agent vs Chain

| Chain | Agent |
|-------|-------|
| Fixed sequence of steps | Dynamic — decides what to do |
| No looping | Can loop until done |
| Deterministic | Non-deterministic |
| Fast, predictable | Flexible, capable |


## 3.1 — Building a ReAct Agent with LangGraph

We'll build a proper agent from scratch — the same architecture used in production AI assistants.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from typing import TypedDict, Annotated, Sequence
import operator, json

# --- Define Tools ---
@tool
def search_wikipedia(query: str) -> str:
    """Search Wikipedia for information about a topic."""
    # Mock — in production, use the wikipedia library
    mock_results = {
        "python": "Python is a high-level programming language known for readability and simplicity.",
        "machine learning": "Machine learning is a subset of AI that enables computers to learn from data without being explicitly programmed.",
        "neural network": "A neural network is a series of algorithms mimicking the human brain to recognize patterns.",
        "transformer": "The Transformer is a deep learning architecture based entirely on attention mechanisms, introduced in the paper 'Attention is All You Need'."
    }
    for key, val in mock_results.items():
        if key in query.lower():
            return val
    return f"No Wikipedia result found for '{query}'."

@tool
def calculate(expression: str) -> str:
    """Evaluate a math expression like '25 * 4 / 2'."""
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"

@tool
def get_current_date() -> str:
    """Get today's date."""
    from datetime import date
    return str(date.today())

tools = [search_wikipedia, calculate, get_current_date]

# --- State Definition ---
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]

# --- Agent Node ---
llm_agent = ChatOllama(model="llama3.1:8b", temperature=0).bind_tools(tools)

def agent_node(state: AgentState) -> dict:
    """The brain: decide what to do."""
    messages = list(state["messages"])
    # Add system message if first turn
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [SystemMessage(content="You are a helpful research assistant. Use tools when needed to answer questions accurately.")] + messages
    
    response = llm_agent.invoke(messages)
    return {"messages": [response]}

# --- Routing ---
def should_use_tools(state: AgentState) -> Literal["tools", END]:
    """Check if the last message has tool calls."""
    last_msg = state["messages"][-1]
    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
        return "tools"
    return END

# --- Build Graph ---
builder = StateGraph(AgentState)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(tools))  # ToolNode auto-executes all tool calls

builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", should_use_tools)
builder.add_edge("tools", "agent")  # After tools, go back to agent

agent = builder.compile()

# --- Run it! ---
def run_agent(question: str, verbose=True):
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    if verbose:
        for msg in result["messages"]:
            if isinstance(msg, HumanMessage):
                print(f"👤 Human: {msg.content}")
            elif isinstance(msg, AIMessage):
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        print(f"🔧 Tool call: {tc['name']}({tc['args']})")
                else:
                    print(f"🤖 Agent: {msg.content}")
            elif isinstance(msg, ToolMessage):
                print(f"📊 Tool result: {msg.content[:100]}")
    return result["messages"][-1].content

answer = run_agent("What is a transformer in machine learning, and what is 2024 - 2017?")
print(f"\n✅ Final: {answer}")

## 3.2 — Multi-Step Research Agent

A more realistic agent that does multi-step research and produces a structured report.

In [ ]:
# Research agent that decomposes a question into sub-tasks

class ResearchState(TypedDict):
    question: str
    sub_questions: list[str]
    findings: list[str]
    current_sub_q_index: int
    final_report: str

def decompose_question(state: ResearchState) -> dict:
    """Break the main question into 3 sub-questions."""
    prompt = f"""Break this question into exactly 3 specific sub-questions that together answer it:
    Question: {state['question']}
    
    Return ONLY a JSON list of 3 strings. Example: ["sub1", "sub2", "sub3"]"""
    
    response = llm.invoke(prompt).content.strip()
    try:
        sub_qs = json.loads(response)
    except:
        sub_qs = [state['question']]  # fallback
    
    print(f"  [decompose] Sub-questions: {sub_qs}")
    return {"sub_questions": sub_qs, "current_sub_q_index": 0, "findings": []}

def research_sub_question(state: ResearchState) -> dict:
    """Answer the current sub-question."""
    idx = state["current_sub_q_index"]
    sub_q = state["sub_questions"][idx]
    
    response = llm.invoke(f"Answer this specifically in 2-3 sentences: {sub_q}")
    finding = f"Q: {sub_q}\nA: {response.content}"
    
    print(f"  [research] Answered sub-question {idx + 1}/{len(state['sub_questions'])}")
    return {
        "findings": state["findings"] + [finding],
        "current_sub_q_index": idx + 1
    }

def has_more_questions(state: ResearchState) -> Literal["research_sub_question", "write_report"]:
    """Route: research next sub-question OR write the final report."""
    if state["current_sub_q_index"] < len(state["sub_questions"]):
        return "research_sub_question"
    return "write_report"

def write_report(state: ResearchState) -> dict:
    """Synthesize findings into a final report."""
    findings_text = "\n\n".join(state["findings"])
    prompt = f"""Write a concise, well-structured report answering: {state['question']}
    
    Based on these findings:
    {findings_text}
    
    Format: Introduction, Key Points, Conclusion."""
    
    report = llm.invoke(prompt).content
    print(f"  [write_report] Final report written")
    return {"final_report": report}

# Build research graph
builder = StateGraph(ResearchState)
builder.add_node("decompose_question", decompose_question)
builder.add_node("research_sub_question", research_sub_question)
builder.add_node("write_report", write_report)

builder.add_edge(START, "decompose_question")
builder.add_conditional_edges("decompose_question", has_more_questions)
builder.add_conditional_edges("research_sub_question", has_more_questions)
builder.add_edge("write_report", END)

research_agent = builder.compile()

result = research_agent.invoke({
    "question": "How does GPT-4 differ from earlier language models and why does it matter?",
    "sub_questions": [],
    "findings": [],
    "current_sub_q_index": 0,
    "final_report": ""
})

print("\n" + "="*60)
print(result["final_report"])

## 3.3 — Multi-Agent System

In complex tasks, we can have **multiple specialized agents** that collaborate — a Supervisor that delegates to sub-agents.

In [ ]:
# Multi-agent system: Supervisor + Specialist Agents

class MultiAgentState(TypedDict):
    task: str
    assigned_agent: str        # "coder", "writer", "analyst"
    agent_output: str
    supervisor_verdict: str
    final_output: str

# Supervisor: decides which agent to use
def supervisor(state: MultiAgentState) -> dict:
    prompt = f"""You are a supervisor. Assign this task to the best specialist:
    - coder: for programming, algorithms, code generation
    - writer: for writing, summarization, creative content
    - analyst: for data analysis, comparisons, research questions
    
    Task: {state['task']}
    Respond ONLY with one word: coder, writer, or analyst."""
    
    assigned = llm.invoke(prompt).content.strip().lower()
    # Clean up
    for opt in ["coder", "writer", "analyst"]:
        if opt in assigned:
            assigned = opt
            break
    else:
        assigned = "writer"  # default fallback
    
    print(f"  [supervisor] Assigned to: {assigned}")
    return {"assigned_agent": assigned}

# Specialist agents
def coder_agent(state: MultiAgentState) -> dict:
    output = llm.invoke(
        f"You are an expert Python developer. Complete this task:\n{state['task']}"
    ).content
    print("  [coder] Code generated")
    return {"agent_output": output}

def writer_agent(state: MultiAgentState) -> dict:
    output = llm.invoke(
        f"You are an expert technical writer. Complete this task:\n{state['task']}"
    ).content
    print("  [writer] Content generated")
    return {"agent_output": output}

def analyst_agent(state: MultiAgentState) -> dict:
    output = llm.invoke(
        f"You are an expert data analyst. Complete this task:\n{state['task']}"
    ).content
    print("  [analyst] Analysis generated")
    return {"agent_output": output}

# Final review
def reviewer(state: MultiAgentState) -> dict:
    prompt = f"""Review this output and give a 1-sentence quality verdict:
    Task: {state['task']}
    Output: {state['agent_output'][:500]}"""
    
    verdict = llm.invoke(prompt).content
    return {
        "supervisor_verdict": verdict,
        "final_output": state["agent_output"]
    }

# Routing function
def route_to_agent(state: MultiAgentState) -> Literal["coder_agent", "writer_agent", "analyst_agent"]:
    return f"{state['assigned_agent']}_agent"

# Build multi-agent graph
builder = StateGraph(MultiAgentState)
builder.add_node("supervisor", supervisor)
builder.add_node("coder_agent", coder_agent)
builder.add_node("writer_agent", writer_agent)
builder.add_node("analyst_agent", analyst_agent)
builder.add_node("reviewer", reviewer)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route_to_agent)
builder.add_edge("coder_agent", "reviewer")
builder.add_edge("writer_agent", "reviewer")
builder.add_edge("analyst_agent", "reviewer")
builder.add_edge("reviewer", END)

multi_agent = builder.compile()

# Test with different task types
tasks = [
    "Write a Python function to find all prime numbers up to N.",
    "Compare the pros and cons of PostgreSQL vs MongoDB for a startup.",
    "Write a short blog post introduction about the rise of AI in healthcare."
]

for task in tasks:
    print(f"\n📋 Task: {task[:60]}...")
    result = multi_agent.invoke({
        "task": task,
        "assigned_agent": "",
        "agent_output": "",
        "supervisor_verdict": "",
        "final_output": ""
    })
    print(f"   Verdict: {result['supervisor_verdict']}")

## 3.4 — Complete Project: AI Tutor Agent

Putting it all together — a full **AI Tutor Agent** that:
- Understands student questions
- Searches for relevant information using tools
- Adapts its explanation based on the student's level
- Generates a quiz to test understanding
- Keeps conversation history across turns

In [ ]:
# Complete AI Tutor Agent

class TutorState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    student_level: str    # "beginner", "intermediate", "advanced"
    topic: str
    mode: str             # "explain", "quiz", "chat"

@tool
def get_concept_explanation(concept: str, level: str) -> str:
    """Get a detailed explanation of an AI/ML concept at a specific level (beginner/intermediate/advanced)."""
    prompt = f"Explain '{concept}' for a {level} student in 3-4 sentences. Use simple analogies for beginners."
    return llm.invoke(prompt).content

@tool
def generate_quiz(topic: str, num_questions: int = 2) -> str:
    """Generate a short quiz on a topic. Returns multiple choice questions."""
    prompt = f"Generate {num_questions} multiple choice questions about '{topic}'. Include correct answer at the end."
    return llm.invoke(prompt).content

@tool
def get_code_example(concept: str) -> str:
    """Get a simple Python code example demonstrating an AI/ML concept."""
    prompt = f"Write a minimal Python code example (under 15 lines) demonstrating '{concept}'."
    return llm.invoke(prompt).content

tutor_tools = [get_concept_explanation, generate_quiz, get_code_example]

tutor_llm = ChatOllama(model="llama3.1:8b", temperature=0.3).bind_tools(tutor_tools)

TUTOR_SYSTEM = """You are an expert AI/ML tutor. You adapt explanations to the student's level.
- Use get_concept_explanation to explain concepts clearly
- Use generate_quiz when the student wants to test themselves
- Use get_code_example when showing code would help
Always be encouraging and patient."""

def tutor_agent(state: TutorState) -> dict:
    msgs = [SystemMessage(content=TUTOR_SYSTEM)] + list(state["messages"])
    response = tutor_llm.invoke(msgs)
    return {"messages": [response]}

def route_tutor(state: TutorState) -> Literal["tools", END]:
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return END

checkpointer = MemorySaver()
builder = StateGraph(TutorState)
builder.add_node("tutor", tutor_agent)
builder.add_node("tools", ToolNode(tutor_tools))
builder.add_edge(START, "tutor")
builder.add_conditional_edges("tutor", route_tutor)
builder.add_edge("tools", "tutor")

tutor = builder.compile(checkpointer=checkpointer)

tutor_config = {"configurable": {"thread_id": "tutor_session_1"}}

def ask_tutor(question: str, level: str = "beginner"):
    result = tutor.invoke({
        "messages": [HumanMessage(content=question)],
        "student_level": level,
        "topic": "",
        "mode": "chat"
    }, config=tutor_config)
    return result["messages"][-1].content

# Demo interaction
print(ask_tutor("Can you explain what a neural network is? I'm completely new to this."))
print("\n" + "-"*60 + "\n")
print(ask_tutor("Can you give me a code example of a simple neural network?"))
print("\n" + "-"*60 + "\n")
print(ask_tutor("Quiz me on what we just covered!"))

---
## 🧠 Part 3 — Exercises

**Exercise 3.1:** Add a 4th tool `recommend_resources(topic: str) -> str` to the ReAct agent that returns learning resources. Test it with: *"What are the best resources for learning about transformers?"*

**Exercise 3.2:** Modify the research agent so that if the final report is shorter than 150 words, it loops back to add more detail. (Hint: add a `check_length` node with a conditional edge.)

**Exercise 3.3:** Add a 4th specialist to the multi-agent system: a `debugger_agent` for error messages and debugging tasks. Update the supervisor prompt accordingly.

**Exercise 3.4 (Challenge):** Build a self-correcting code agent that: (1) Generates Python code for a task, (2) Tries to execute it, (3) If it fails, feeds the error back and tries to fix it, (4) Loops up to 3 times. Use `exec()` with try/catch to simulate code execution.

In [ ]:
# Your Exercise 3.1 code here


In [ ]:
# Your Exercise 3.2 code here


In [ ]:
# Your Exercise 3.3 code here


In [ ]:
# Your Exercise 3.4 (Challenge) code here


---
# 🗺️ Concept Map & Summary

```
                      ┌─────────────────────────────────┐
                      │         AGENTIC AI STACK         │
                      └─────────────────────────────────┘
                                     │
             ┌───────────────────────┼───────────────────────┐
             ▼                       ▼                       ▼
     ┌──────────────┐       ┌──────────────┐       ┌──────────────┐
     │  LANGCHAIN   │       │  LANGGRAPH   │       │   AGENTS     │
     │              │       │              │       │              │
     │ • LLMs       │       │ • Nodes      │       │ • ReAct      │
     │ • Prompts    │       │ • Edges      │       │ • Multi-Agent│
     │ • Chains     │       │ • State      │       │ • Supervisor │
     │ • Memory     │       │ • Loops      │       │ • Tool Use   │
     │ • Tools      │       │ • Branching  │       │ • Self-Corr. │
     │ • LCEL (|)   │       │ • Checkpoint │       │ • Memory     │
     └──────────────┘       └──────────────┘       └──────────────┘
```

## Key Takeaways

| | LangChain | LangGraph | Agentic AI |
|---|-----------|-----------|------------|
| **Use when** | Building pipelines | Need loops/branches | Full autonomy needed |
| **Core abstraction** | Chain (`A|B|C`) | Graph (nodes + edges) | Agent (think+act loop) |
| **State** | Passed through chain | Shared TypedDict | Message history |
| **Memory** | Message history objects | Checkpointers | Persisted state |
| **Complexity** | Low–Medium | Medium–High | High |

## What to Learn Next

- **RAG (Retrieval-Augmented Generation):** Connect agents to vector databases
- **LangSmith:** Tracing and debugging LangChain/LangGraph applications  
- **Streaming:** Stream agent thoughts and responses in real-time
- **Human-in-the-loop:** Pause graphs for human approval before continuing
- **Production deployment:** FastAPI + LangGraph for deployed agents